# STOCHASTIC GRADIENT DESCENT (SGD)

## 1. Mathematical Foundation

In traditional Batch Gradient Descent (BGD), parameters are updated by computing the gradient of the cost function across the entire training dataset of size $m$. While mathematically stable, this approach becomes computationally prohibitive for massive datasets.

Stochastic Gradient Descent (SGD) addresses this bottleneck by performing a parameter update for *each individual training example*. Assuming the dataset is randomly shuffled, the update rule at iteration $t$ for a single sample $(x^{(i)}, y^{(i)})$ is defined as:

$$W^{(t+1)} = W^{(t)} - \alpha \cdot \nabla_W L(W^{(t)}; x^{(i)}, y^{(i)})$$

Where $\alpha$ denotes the learning rate, and $L$ represents the loss function computed on a single data point.

## 2. Optimization Landscape Characteristics

Estimating the true gradient using a single sample fundamentally alters the optimization dynamics:

* **Noisy Convergence:** Geometrically, the path taken by SGD across the loss landscape is not a smooth descent. Because the gradient of individual samples exhibits high variance, the trajectory oscillates heavily around the optimal path towards the minimum.
* **Escaping Local Minima:** This inherent noise acts as an implicit regularization mechanism. The erratic jumps allow the algorithm to escape narrow local minima and saddle points, which are prevalent in high-dimensional, non-convex optimization spaces.
* **Asymptotic Behavior:** Due to continuous oscillation, pure SGD never truly converges to the exact global or local minimum. It will perpetually wander around the minimum region. Practitioners often employ *Learning Rate Decay* to gradually reduce $\alpha$, allowing the algorithm to settle closer to the exact minimum over time.

## 3. Computational Limitations

Despite solving the memory constraints of BGD, pure SGD introduces a critical hardware inefficiency. By processing $m=1$ sequentially, the algorithm completely breaks the matrix computation paradigm. It fails to utilize the parallel processing capabilities (Vectorization) of modern GPUs, leading to severe computational waste. This exact limitation is the primary motivation for the widespread adoption of **Mini-batch Gradient Descent**.

## 4. Practical Implementations

Below are the complete, runnable implementations of pure Stochastic Gradient Descent (SGD). The critical technical requirement here is enforcing a strict `batch_size=1` constraint during data ingestion, ensuring that the model parameters are updated after every single sample.

### 4.1. PyTorch Implementation
In PyTorch, pure SGD is implemented by combining the `optim.SGD` module (with momentum set to 0) and configuring the `DataLoader` to yield exactly one sample per iteration.

~~~python
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Generate dummy dataset (1,000 samples, 10 features)
X = torch.randn(1000, 10)
Y = torch.randn(1000, 1)
dataset = TensorDataset(X, Y)

# 2. PURE SGD DATA CONFIGURATION: batch_size MUST be exactly 1
# Shuffling is mandatory to prevent cyclical oscillations
sgd_loader = DataLoader(dataset, batch_size=1, shuffle=True)

# 3. Model, Loss, and Optimizer Setup
model = nn.Linear(in_features=10, out_features=1)
criterion = nn.MSELoss()
# Pure SGD: Momentum must be 0.0
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.0)

# 4. Operational Training Loop
epochs = 3
for epoch in range(epochs):
    epoch_loss = 0.0
    
    # In pure SGD, this loop iterates exactly m times (1000 times)
    for x_i, y_i in sgd_loader:
        
        # Step A: Reset gradients
        optimizer.zero_grad()
        
        # Step B: Forward propagation on a SINGLE sample
        outputs = model(x_i)
        
        # Step C: Compute the loss
        loss = criterion(outputs, y_i)
        
        # Step D: Backward propagation
        loss.backward()
        
        # Step E: Update weights immediately
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} completed. Average Loss: {epoch_loss/len(sgd_loader):.4f}")
~~~

### 4.2. TensorFlow / Keras Implementation
In TensorFlow, the Keras API provides a highly abstracted `model.fit` method. To force the framework to execute pure SGD instead of its default mini-batch behavior, the `batch_size` argument must be explicitly overridden to 1.

~~~python
import tensorflow as tf
import numpy as np

# 1. Generate dummy dataset (1,000 samples, 10 features)
X = np.random.randn(1000, 10).astype(np.float32)
Y = np.random.randn(1000, 1).astype(np.float32)

# 2. Model Setup
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=(10,))
])

# 3. Initialize Pure Stochastic Gradient Descent
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.0)

# Compile the model
model.compile(optimizer=optimizer, loss='mse')

# 4. Operational Training Execution
# Enforcing batch_size=1 overrides the default batching (usually 32)
# and applies the gradient update sample-by-sample.
print("Starting training with pure Stochastic Gradient Descent...")
model.fit(
    x=X, 
    y=Y, 
    batch_size=1, 
    epochs=3,
    shuffle=True # Mandatory for SGD stability
)
~~~

# MINI-BATCH GRADIENT DESCENT

## 1. Mathematical Foundation

Batch Gradient Descent (BGD) computes the gradient over the entire dataset (which is computationally expensive and slow), while Stochastic Gradient Descent (SGD) computes it over a single sample (which is highly noisy and inefficient for GPUs). **Mini-batch Gradient Descent** serves as the optimal compromise by dividing the training dataset into smaller, manageable chunks called *mini-batches* of size $B$.

The gradient of a mini-batch is the average of the gradients of its $B$ constituent samples, acting as an unbiased estimator of the true gradient. The update rule for the $t$-th mini-batch is defined as:

$$dW^{[t]} = \frac{1}{B} \sum_{i=1}^{B} \nabla_W L(W; x^{(i)}, y^{(i)})$$
$$W = W - \alpha \cdot dW^{[t]}$$

## 2. Hardware Efficiency and Memory Alignment

In practice, the mini-batch size $B$ is almost always chosen as a power of 2 (e.g., 32, 64, 128, 256, 512). This is not a mathematical requirement, but a hardware-driven architectural necessity.

Modern Graphical Processing Units (GPUs) organize physical memory and streaming multiprocessors in binary structures. Providing a matrix size that aligns with these powers of 2 ensures perfect memory alignment, maximizing parallel processing throughput (Vectorization) and preventing wasted compute cycles.

## 3. Disambiguation: Epoch vs. Iteration

When transitioning to Mini-batch Gradient Descent, maintaining precise control over the training loop requires a strict distinction between an *Epoch* and an *Iteration* (Step):

* **Iteration (Step):** A single execution of the forward and backward propagation passes on exactly *one* mini-batch, resulting in a single weight update.
* **Epoch:** One complete pass through the *entire* training dataset.

**Mathematical Implication:** Assume a dataset of $m = 10,000$ samples and a chosen batch size of $B = 1,000$. To complete 1 Epoch, the algorithm must perform $\frac{10,000}{1,000} = 10$ Iterations. Consequently, the model parameters ($W, b$) are updated 10 times per Epoch.

## 4. The Two-Step Partitioning Algorithm

In empirical implementations, one cannot simply slice the dataset sequentially. If the data is ordered by class (e.g., the first 1,000 images are Class A, the next 1,000 are Class B), a sequential slice will induce severe local bias. The network would optimize exclusively for one class per iteration, completely destabilizing the convergence trajectory.

The industry-standard preparation pipeline mandates a strict two-step process per epoch:

1. **Step 1: Synchronous Shuffling.** The columns of the feature matrix $X$ and the label matrix $Y$ must be randomly shuffled in unison. This destroys any pre-existing ordering and ensures that each mini-batch is a statistically representative subset of the overall data distribution.
2. **Step 2: Partitioning.** The shuffled matrices are sliced into discrete blocks of size $B$. If the total number of samples $m$ is not perfectly divisible by $B$, the algorithm dynamically creates a final, smaller mini-batch containing the residual samples (size $< B$) to ensure no data is discarded.

## 5. Strategic Comparison of Gradient Descent Variants

| Criteria | Batch GD (BGD) | Stochastic GD (SGD) | Mini-batch GD |
| :--- | :--- | :--- | :--- |
| **Sample Size** | $B = m$ (Full Dataset) | $B = 1$ | $1 < B < m$ (e.g., 64, 128) |
| **Vectorization** | Maximum. Fully utilizes GPU. | Negligible. Serial computation. | High. Optimizes matrix operations. |
| **Step Velocity** | Extremely slow per step. | Extremely fast per step. | Balanced computational time. |
| **Convergence Path** | Smooth, direct descent to minimum. | Highly erratic, zig-zag trajectory. | Oscillatory, but with strong directional momentum. |
| **Local Minima Escape**| Poor (Prone to getting stuck). | Excellent (Due to high variance/noise). | Good (Provides sufficient stochasticity). |
| **Primary Use Case** | Small datasets ($m \le 2000$). | Online/Streaming real-time data. | **Industry standard** for Deep Learning. |

---

## 6. Practical Implementations

Below are the complete, runnable implementations of Mini-batch Gradient Descent, demonstrating how modern frameworks handle shuffling, partitioning, and the step-by-step optimization loop.

### 6.1. PyTorch Implementation
PyTorch automates the two-step partitioning algorithm through the `DataLoader` class. 

~~~python
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Generate dummy dataset (10,000 samples, 20 features)
X = torch.randn(10000, 20)
Y = torch.randint(0, 2, (10000, 1)).float()
dataset = TensorDataset(X, Y)

# 2. MINI-BATCH LOADER: Handles Shuffling (Step 1) and Partitioning (Step 2)
batch_size = 64
mini_batch_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# 3. Model, Loss, and Optimizer Setup
model = nn.Linear(in_features=20, out_features=1)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 4. Training Loop
epochs = 5
for epoch in range(epochs):
    epoch_loss = 0.0
    
    # Each iteration in this loop processes exactly ONE mini-batch
    for batch_X, batch_Y in mini_batch_loader:
        
        # Step A: Zero the gradients from the previous iteration
        optimizer.zero_grad()
        
        # Step B: Forward propagation
        outputs = model(batch_X)
        
        # Step C: Compute the cost
        loss = criterion(outputs, batch_Y)
        
        # Step D: Backward propagation (calculate gradients)
        loss.backward()
        
        # Step E: Update parameters
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} completed. Average Loss: {epoch_loss/len(mini_batch_loader):.4f}")
~~~

### 6.2. TensorFlow / Keras Implementation (Custom Training Loop)
TensorFlow leverages the `tf.data.Dataset` API to build a robust data pipeline, and `tf.GradientTape` to manually control the forward and backward passes for each mini-batch.

~~~python
import tensorflow as tf
import numpy as np

# 1. Generate dummy dataset
X = np.random.randn(10000, 20).astype(np.float32)
Y = np.random.randint(0, 2, (10000, 1)).astype(np.float32)

# 2. DATA PIPELINE: Handles Shuffling (Step 1) and Partitioning (Step 2)
batch_size = 64
dataset = tf.data.Dataset.from_tensor_slices((X, Y))
# Buffer size should ideally be >= dataset size for perfect shuffling
mini_batch_dataset = dataset.shuffle(buffer_size=10000).batch(batch_size)

# 3. Model, Loss, and Optimizer Setup
model = tf.keras.Sequential([tf.keras.layers.Dense(1)])
loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)

# 4. Training Loop
epochs = 5
for epoch in range(epochs):
    epoch_loss = 0.0
    steps = 0
    
    # Each iteration processes exactly ONE mini-batch
    for batch_X, batch_Y in mini_batch_dataset:
        
        # Step A & B: Record operations for automatic differentiation (Forward Pass)
        with tf.GradientTape() as tape:
            outputs = model(batch_X, training=True)
            loss = loss_fn(batch_Y, outputs)
            
        # Step C: Compute gradients (Backward Pass)
        gradients = tape.gradient(loss, model.trainable_variables)
        
        # Step D: Update parameters
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        
        epoch_loss += loss.numpy()
        steps += 1
        
    print(f"Epoch {epoch+1}/{epochs} completed. Average Loss: {epoch_loss/steps:.4f}")
~~~

# EXPONENTIALLY WEIGHTED AVERAGES (EWA)

## 1. Mathematical Foundation and Intuition

Exponentially Weighted Averages (EWA) is a statistical technique used to smooth out noisy data sequences by prioritizing recent observations while maintaining a "memory" of past values.

The general recursive formula at time step $t$ is:

$$V_t = \beta V_{t-1} + (1 - \beta) \theta_t$$

Where:
* $V_t$: The smoothed value at the current step.
* $V_{t-1}$: The smoothed value from the previous step.
* $\theta_t$: The actual observed value at the current step.
* $\beta$: The momentum hyperparameter ($0 \le \beta < 1$).



## 2. Mathematical Proofs

### 2.1. Why $V_t$ averages over approximately $1/(1-\beta)$ steps?

To understand the "memory window," we expand the recursive formula into a power series:
$$V_t = (1 - \beta) \theta_t + (1 - \beta)\beta \theta_{t-1} + (1 - \beta)\beta^2 \theta_{t-2} + \dots + (1 - \beta)\beta^{k} \theta_{t-k}$$

The weight assigned to an observation from $k$ steps ago is $w(k) = (1 - \beta)\beta^k$. We define the effective window limit $k$ as the point where the weight drops to $1/e$ (approx. 36.8%) of its initial value:
$$\beta^k = \frac{1}{e}$$

Taking the natural logarithm on both sides:
$$k \ln(\beta) = -1 \implies k = \frac{-1}{\ln(\beta)}$$

Using the Taylor expansion for $\ln(\beta)$ when $\beta$ is close to 1: $\ln(\beta) \approx -(1 - \beta)$. Substituting this:
$$k \approx \frac{-1}{-(1 - \beta)} = \frac{1}{1 - \beta}$$

**Conclusion:** The EWA value $V_t$ essentially represents the average of the most recent $1/(1-\beta)$ data points.



### 2.2. Proof and Derivation of Bias Correction

At the start of the sequence ($t=1$), $V_0=0$ causes $V_t$ to be heavily biased toward zero. To derive the correction factor, assume all $\theta_i$ are approximately equal to a constant $C$.
The uncorrected values are:
* $V_1 = (1 - \beta)C$
* $V_2 = \beta V_1 + (1 - \beta)C = \beta(1 - \beta)C + (1 - \beta)C = (1 - \beta^2)C$
* $V_3 = \beta V_2 + (1 - \beta)C = (1 - \beta^3)C$

By induction, the general form for $V_t$ is:
$$V_t = (1 - \beta^t)C$$

To recover the true value $C$, we must divide $V_t$ by $(1 - \beta^t)$. Thus, the **Bias Corrected** value $\hat{V}_t$ is:
$$\hat{V}_t = \frac{V_t}{1 - \beta^t}$$

As $t \to \infty$, $\beta^t \to 0$, and the correction factor $1/(1-\beta^t) \to 1$, naturally deactivating the correction.

## 3. Practical Implementations

### 3.1. PyTorch Implementation (Manual Calculation)

~~~python
import torch
import matplotlib.pyplot as plt

# 1. Generate noisy dummy data
t = torch.linspace(0, 100, 100)
theta = 30 + 5 * torch.sin(t / 5) + torch.randn(100)

def compute_ewa(data, beta=0.9, apply_bias_correction=True):
    v = 0
    ewa_values = []
    for i, theta_t in enumerate(data):
        v = beta * v + (1 - beta) * theta_t
        if apply_bias_correction:
            v_corrected = v / (1 - beta ** (i + 1))
            ewa_values.append(v_corrected)
        else:
            ewa_values.append(v)
    return torch.tensor(ewa_values)

# 2. Visualize the effect
v_smooth = compute_ewa(theta, beta=0.9)
v_very_smooth = compute_ewa(theta, beta=0.98)

plt.plot(theta.numpy(), label='Raw Data', alpha=0.3)
plt.plot(v_smooth.numpy(), label='Beta=0.9 (10 steps)')
plt.plot(v_very_smooth.numpy(), label='Beta=0.98 (50 steps)')
plt.legend()
plt.title("EWA Smoothing with Proof-based Logic")
plt.show()
~~~

### 3.2. TensorFlow / Keras Implementation

~~~python
import tensorflow as tf
import numpy as np

# Use ExponentialMovingAverage for shadow variables
variable = tf.Variable(30.0, name="temp")
ema = tf.train.ExponentialMovingAverage(decay=0.9)

# Update the variable and apply EWA
new_value = 35.0
variable.assign(new_value)
ema.apply([variable])

# Retrieve the smoothed value
smoothed_val = ema.average(variable)
print(f"Current Value: {variable.numpy()}")
print(f"Exponential Moving Average: {smoothed_val.numpy()}")
~~~

# GRADIENT DESCENT WITH MOMENTUM

## 1. The Pathological Curvature Problem

To understand the necessity of Momentum, one must analyze the failure modes of standard Mini-batch or Stochastic Gradient Descent (SGD) in complex optimization landscapes. 

Standard SGD struggles significantly when navigating areas of **pathological curvature**—specifically, steep ravines or elongated valleys where the surface curves much more steeply in one dimension than in another. 
* Along the steep axis (let's denote it as $b$), the gradient is exceptionally large.
* Along the shallow axis pointing towards the local/global minimum (let's denote it as $W$), the gradient is relatively small.

As standard SGD updates parameters proportionally to the immediate gradient, it takes massive, erratic jumps back and forth across the steep ravine (oscillation), while making agonizingly slow progress along the shallow path toward the minimum. Increasing the learning rate ($\alpha$) to speed up horizontal progress only exacerbates the vertical oscillations, often leading to divergence.



## 2. Mathematical Intervention: Exponentially Weighted Averages

Momentum resolves this inefficiency by applying an **Exponentially Weighted Average (EWA)** directly to the gradients before performing the parameter update. Instead of updating weights based strictly on the current gradient step, the algorithm computes a "velocity" vector.

The update equations at iteration $t$ are:

1. **Compute Velocity:**
   $$v_{dW}^{[t]} = \beta v_{dW}^{[t-1]} + (1 - \beta) dW^{[t]}$$
   $$v_{db}^{[t]} = \beta v_{db}^{[t-1]} + (1 - \beta) db^{[t]}$$

2. **Update Parameters:**
   $$W^{[t]} = W^{[t-1]} - \alpha v_{dW}^{[t]}$$
   $$b^{[t]} = b^{[t-1]} - \alpha v_{db}^{[t]}$$

Here, $\beta$ is the momentum hyperparameter (commonly set to $0.9$), and $v$ represents the moving average of the gradients.

## 3. Vector Cancellation and Accumulation Analysis

The elegance of Momentum lies in how the EWA behaves across the two distinct geometric axes:

### 3.1. The Steep Axis (Oscillation Cancellation)
Along the steep vertical axis ($b$), the algorithm oscillates. The gradient $db$ rapidly alternates signs at each iteration (e.g., positive, then negative, then positive). 
When passed through the EWA equation, these alternating opposite vectors sum together and **cancel each other out** (destructive interference). Consequently, the velocity term $v_{db}$ approaches $0$, effectively dampening the vertical oscillations.

### 3.2. The Shallow Axis (Velocity Accumulation)
Along the horizontal axis ($W$) pointing towards the minimum, the gradient $dW$ consistently points in the same direction (same sign) across multiple iterations.
When passed through the EWA equation, these consistent vectors **accumulate** (constructive interference). The velocity term $v_{dW}$ steadily grows, acting like a snowball rolling down a hill. This dramatically accelerates the algorithm's progress along the optimal path.

## 4. Physical Intuition

The algorithm heavily borrows terminology from classical mechanics to provide intuition:
* The current gradient ($dW, db$) acts as **Acceleration**, providing a force that modifies the current trajectory.
* The EWA term ($v_{dW}, v_{db}$) acts as **Velocity**, representing the accumulated momentum of the parameter particle moving through the loss landscape.
* The hyperparameter $\beta$ acts as **Friction** or **Air Resistance**. Because $\beta < 1$, it continuously decays the historical velocity, preventing the particle from accelerating infinitely and overshooting the minimum.

## 5. Practical Implementations

Below are runnable implementations of Momentum. *Note on Framework Variations:* While Andrew Ng's mathematical formulation includes the $(1-\beta)$ scaling factor, deep learning frameworks like PyTorch strictly implement Momentum as $v_{dW} = \beta v_{dW} + dW$. Mathematically, this omitted scaling is seamlessly absorbed by tuning the learning rate $\alpha$.

### 5.1. PyTorch Implementation

~~~python
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Dataset setup
X = torch.randn(1000, 10)
Y = torch.randn(1000, 1)
loader = DataLoader(TensorDataset(X, Y), batch_size=64, shuffle=True)

# 2. Model setup
model = nn.Linear(in_features=10, out_features=1)
criterion = nn.MSELoss()

# 3. SGD with Momentum Initialization
# Setting momentum > 0 activates the exponentially weighted average logic
beta = 0.9
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=beta)

# 4. Training Loop
for epoch in range(5):
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        # The optimizer internally computes velocity and updates weights
        optimizer.step()
~~~

### 5.2. TensorFlow / Keras Implementation

~~~python
import tensorflow as tf
import numpy as np

# 1. Dataset setup
X = np.random.randn(1000, 10).astype(np.float32)
Y = np.random.randn(1000, 1).astype(np.float32)

# 2. Model setup
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=(10,))
])

# 3. SGD with Momentum Initialization
# TensorFlow allows explicit declaration of the momentum hyperparameter
beta = 0.9
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=beta)

model.compile(optimizer=optimizer, loss='mse')

# 4. Training Execution
model.fit(X, Y, batch_size=64, epochs=5, verbose=1)
~~~

# ADAGRAD (ADAPTIVE GRADIENT ALGORITHM)

## 1. The Paradigm Shift: Per-Parameter Learning Rates

Prior to AdaGrad (introduced by Duchi et al. in 2011), optimization algorithms like standard SGD or Momentum applied a single, global learning rate ($\alpha$) to all parameters simultaneously. 

AdaGrad introduced a paradigm shift: **Adaptive Learning Rates**. It scales the learning rate individually for every single parameter in the network based on its historical gradients.
* **Infrequent parameters** (associated with rare features) receive **larger** updates to learn quickly when they finally appear.
* **Frequent parameters** (associated with common features) receive **smaller** updates to fine-tune their values without destabilizing the model.

This makes AdaGrad exceptionally powerful for dealing with **sparse data**, such as Natural Language Processing (NLP) tasks or recommendation systems where certain words or items appear very rarely.

## 2. Mathematical Foundation

Instead of using an Exponentially Weighted Average (EWA) like Momentum or RMSprop, AdaGrad uses a strict **Accumulator**. It keeps a running sum of the squared gradients for each parameter from the very first iteration.

Let $dW^{[t]}$ be the gradient at iteration $t$. The algorithm maintains an accumulator $s_{dW}$:

1. **Gradient Accumulation:**
   $$s_{dW}^{[t]} = s_{dW}^{[t-1]} + (dW^{[t]})^2$$
   $$s_{db}^{[t]} = s_{db}^{[t-1]} + (db^{[t]})^2$$
   *(Note: The squaring is applied element-wise. All values in $s_{dW}$ are strictly positive and monotonically increasing).*

2. **Parameter Update:**
   $$W^{[t]} = W^{[t-1]} - \frac{\alpha}{\sqrt{s_{dW}^{[t]}} + \epsilon} \odot dW^{[t]}$$
   $$b^{[t]} = b^{[t-1]} - \frac{\alpha}{\sqrt{s_{db}^{[t]}} + \epsilon} \odot db^{[t]}$$

Where:
* $\alpha$: The initial, global learning rate.
* $\epsilon$: A tiny smoothing term (e.g., $10^{-8}$) to prevent division by zero.
* $\odot$: Element-wise multiplication.

### 2.1. The Adaptive Mechanism Explained
Look at the denominator $\sqrt{s_{dW}^{[t]}}$. 
If a specific weight $W_i$ has had very large gradients in the past, its corresponding accumulator $s_{dW, i}$ will be massive. Consequently, the effective learning rate $\frac{\alpha}{\sqrt{s_{dW, i}}}$ for that specific weight shrinks dramatically. Conversely, a weight with small historical gradients maintains a higher effective learning rate.

## 3. The Fatal Flaw: Premature Decay (Vanishing Learning Rate)

While conceptually brilliant for sparse convex problems, AdaGrad harbors a fatal mathematical flaw when applied to non-convex Deep Learning optimization (like training deep neural networks).

Because $s_{dW}$ is formed by adding strictly positive squared numbers ($(dW)^2$), the accumulator **grows monotonically** towards infinity as training progresses ($t \to \infty$).
Consequently, the effective learning rate strictly decreases:
$$\lim_{t \to \infty} \frac{\alpha}{\sqrt{s_{dW}^{[t]}}} = 0$$

**The Impact:** Deep learning models require prolonged navigation through complex, non-convex loss landscapes. AdaGrad's learning rate shrinks so rapidly that the algorithm effectively "stops learning" long before it reaches the global (or a good local) minimum. The network freezes prematurely.

*This exact mathematical flaw is what prompted Geoffrey Hinton to invent **RMSprop**, which replaces the infinite sum with a decaying Exponentially Weighted Average (EWA) to prevent the denominator from exploding.*

## 4. Practical Implementations

Despite its flaw in deep networks, AdaGrad is still actively used in specific sparse-data domains. Below are the runnable implementations in modern frameworks.

### 4.1. PyTorch Implementation

~~~python
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Dataset setup (Simulating a sparse dataset scenario)
X = torch.randn(1000, 50)
Y = torch.randint(0, 2, (1000, 1)).float()
loader = DataLoader(TensorDataset(X, Y), batch_size=32, shuffle=True)

# 2. Model setup
model = nn.Linear(in_features=50, out_features=1)
criterion = nn.BCEWithLogitsLoss()

# 3. AdaGrad Initialization
# Note: lr_decay is usually kept at 0 to rely purely on the adaptive mechanism,
# weight_decay can be used for L2 regularization.
optimizer = optim.Adagrad(model.parameters(), lr=0.01, lr_decay=0, weight_decay=0, eps=1e-10)

# 4. Training Loop
for epoch in range(5):
    epoch_loss = 0.0
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        
        # AdaGrad internally accumulates the squared gradients and scales the step
        optimizer.step()
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(loader):.4f}")
~~~

### 4.2. TensorFlow / Keras Implementation

~~~python
import tensorflow as tf
import numpy as np

# 1. Dataset setup
X = np.random.randn(1000, 50).astype(np.float32)
Y = np.random.randint(0, 2, (1000, 1)).astype(np.float32)

# 2. Model setup
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=(50,))
])

# 3. AdaGrad Initialization
# epsilon is crucial to avoid division by zero.
optimizer = tf.keras.optimizers.Adagrad(
    learning_rate=0.01,
    initial_accumulator_value=0.1, # Gives a slight warm-up delay to the decay
    epsilon=1e-07
)

model.compile(optimizer=optimizer, loss='binary_crossentropy')

# 4. Training Execution
model.fit(X, Y, batch_size=32, epochs=5, verbose=1)
~~~

# RMSPROP (ROOT MEAN SQUARE PROPAGATION)

## 1. The Core Motivation: Fixing AdaGrad's Fatal Flaw

AdaGrad achieves per-parameter adaptive learning rates by accumulating the squared gradients. However, its fatal flaw in non-convex Deep Learning optimization is that the accumulator strictly grows to infinity, forcing the effective learning rate to zero and halting the learning process prematurely.

**RMSprop** solves this by fundamentally changing how the historical squared gradients are stored. Instead of an infinite sum, it uses an **Exponentially Weighted Average (EWA)** of the squared gradients. This mechanism ensures that the algorithm "forgets" the distant past and only focuses on the volatility of the most recent iterations.

## 2. The Role of $\beta$: Memory and Recovery (AdaGrad vs. RMSprop)

To understand the profound impact of RMSprop, one must analyze the hyperparameter $\beta$ through the lens of algorithmic "Memory" and "Recovery."

### 2.1. The Limitation of AdaGrad: "Infinite Memory"
In AdaGrad, the accumulator equation is $s_{dW} = s_{dW} + (dW)^2$.
* **The Implication:** AdaGrad possesses an infinite memory. Every squared gradient from the first epoch permanently inflates the denominator $\left(\frac{\alpha}{\sqrt{s_{dW}}}\right)$.
* **The Consequence:** The effective learning rate is a one-way street—it can only decrease. If the model traverses a highly volatile, steep region (a "ravine"), $s_{dW}$ accumulates massive values, crushing the learning rate. If the model subsequently escapes into a flat region (a "plateau") where larger steps are necessary to maintain momentum, AdaGrad remains paralyzed by its historical penalty.

### 2.2. The Salvation of RMSprop: "The Right to Forget"
Geoffrey Hinton recognized that in the highly non-convex loss landscapes of Deep Learning, the algorithm must discard distant history. By introducing $\beta$ via EWA:
$$s_{dW} = \beta s_{dW} + (1 - \beta) (dW)^2$$
* **Limiting the Memory Window:** As proven in EWA mathematics, $\beta$ restricts the algorithm's memory to approximately the last $\frac{1}{1-\beta}$ steps.
* **Exponential Forgetting:** Massive gradients from early epochs are continuously multiplied by $\beta^t$. As $t \to \infty$, these early spikes decay to zero, effectively erasing the "ravine" from the algorithm's memory.

### 2.3. The Magic of Recovery (Local Adaptation)
The critical superiority of RMSprop is **Recovery**. 
When transitioning from a steep ravine to a flat plateau, the gradient $dW$ shrinks. Because of the EWA decay, the previously massive $s_{dW}$ is multiplied by $\beta$ (e.g., $0.9$), actively shrinking it by 10% per step. The new additions $(1-\beta)(dW)^2$ are too small to offset this decay.
* **The Result:** The denominator $\sqrt{s_{dW}}$ dynamically *shrinks*, which actively *increases* the effective learning rate $\frac{\alpha}{\sqrt{s_{dW}}}$. 

RMSprop transforms the rigid decay of AdaGrad into an adaptive suspension system: it decelerates in high-volatility regions and automatically re-accelerates in flat regions.

## 3. Mathematical Foundation

RMSprop maintains two independent state matrices/vectors ($s_{dW}$ and $s_{db}$) for the weights and biases. At iteration $t$:

**Step 1: Compute the EWA of Squared Gradients**
$$s_{dW}^{[t]} = \beta s_{dW}^{[t-1]} + (1 - \beta) (dW^{[t]})^2$$
$$s_{db}^{[t]} = \beta s_{db}^{[t-1]} + (1 - \beta) (db^{[t]})^2$$
*(Note: The squaring is element-wise. $\beta$ is the decay hyperparameter, often set to $0.9$. In some texts, this specific $\beta$ is denoted as $\beta_2$ or $\rho$).*

**Step 2: Update Parameters (The Root Mean Square step)**
$$W^{[t]} = W^{[t-1]} - \frac{\alpha}{\sqrt{s_{dW}^{[t]}} + \epsilon} \odot dW^{[t]}$$
$$b^{[t]} = b^{[t-1]} - \frac{\alpha}{\sqrt{s_{db}^{[t]}} + \epsilon} \odot db^{[t]}$$

Where:
* $\alpha$: The global base learning rate.
* $\epsilon$: A small constant (e.g., $10^{-8}$) to ensure numerical stability.
* $\odot$: Element-wise multiplication.

## 4. Geometric Interpretation and The "R-M-S" Breakdown

The name RMSprop perfectly describes its mathematical sequence:
1. **Square (S):** It squares the gradients ($(dW)^2, (db)^2$) to discard directional signs, extracting pure volatility.
2. **Mean (M):** It computes the moving average of these squared values using EWA ($\beta$).
3. **Root (R):** It takes the square root ($\sqrt{s}$) in the denominator to scale the updates back to the original unit proportions.

### 4.1. Damping Pathological Curvature
* **Along the steep vertical axis ($b$):** The gradients $db$ are massive. Consequently, $(db)^2$ is huge, making $s_{db}$ large. Dividing by a large $\sqrt{s_{db}}$ severely reduces the effective learning rate for $b$, **damping the oscillations**.
* **Along the shallow horizontal axis ($W$):** The gradients $dW$ are small. Consequently, $(dW)^2$ is small, keeping $s_{dW}$ relatively small. Dividing by a small $\sqrt{s_{dW}}$ ensures the effective learning rate for $W$ remains large, **maintaining forward velocity**.

Unlike Momentum (which cancels out alternating directions in the numerator), RMSprop actively scales the landscape space itself by normalizing the denominator.

## 5. Practical Implementations

Below are the standard, runnable implementations of RMSprop.

### 5.1. PyTorch Implementation

~~~python
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Dataset setup
X = torch.randn(1000, 20)
Y = torch.randint(0, 2, (1000, 1)).float()
loader = DataLoader(TensorDataset(X, Y), batch_size=64, shuffle=True)

# 2. Model setup
model = nn.Linear(in_features=20, out_features=1)
criterion = nn.BCEWithLogitsLoss()

# 3. RMSprop Initialization
# alpha (lr) = 0.001, beta (alpha in PyTorch args) = 0.99, epsilon = 1e-08
# Note: PyTorch confusingly names the EWA decay parameter 'alpha' instead of beta/rho.
optimizer = optim.RMSprop(model.parameters(), lr=0.001, alpha=0.99, eps=1e-08)

# 4. Training Loop
for epoch in range(5):
    epoch_loss = 0.0
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        
        # RMSprop computes the moving average of squared gradients and scales updates
        optimizer.step()
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(loader):.4f}")
~~~

### 5.2. TensorFlow / Keras Implementation

~~~python
import tensorflow as tf
import numpy as np

# 1. Dataset setup
X = np.random.randn(1000, 20).astype(np.float32)
Y = np.random.randint(0, 2, (1000, 1)).astype(np.float32)

# 2. Model setup
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=(20,))
])

# 3. RMSprop Initialization
# TensorFlow uses 'rho' to represent the EWA decay factor (\beta)
optimizer = tf.keras.optimizers.RMSprop(
    learning_rate=0.001,
    rho=0.9,
    epsilon=1e-07
)

model.compile(optimizer=optimizer, loss='binary_crossentropy')

# 4. Training Execution
model.fit(X, Y, batch_size=64, epochs=5, verbose=1)
~~~

# ADAM (ADAPTIVE MOMENT ESTIMATION)

## 1. The Design Philosophy: Integrating Optimization Paradigms

Introduced by Diederik P. Kingma and Jimmy Ba in 2014, the Adam algorithm represents the convergence of two modern optimization paradigms in Deep Learning. Its core objective is to construct an optimizer capable of navigating highly non-convex loss landscapes robustly, while effectively handling both sparse gradients and high-noise objective functions.

Adam achieves this by seamlessly integrating:
1. **Inertial Dynamics (From Momentum):** It preserves historical gradient information via a first-order moment to traverse flat plateaus and dampen local oscillations.
2. **Adaptive Scaling (From RMSprop):** It dynamically scales the learning rate for each individual parameter based on a second-order moment to decelerate safely in regions of pathological curvature.

## 2. Mathematical Foundation and Algorithmic Anatomy

Adam maintains independent state memory for every parameter in the neural network. Assuming that at iteration $t$, the computed gradients of the loss function with respect to the weights and biases are $dW^{[t]}$ and $db^{[t]}$, the update mechanism executes four rigorous mathematical steps:

### Step 1: First Moment Estimation (Gradient Mean)
The algorithm computes the Exponentially Weighted Average (EWA) of the gradients. This acts as a "velocity" vector, dictating the optimal trajectory:
$$v_{dW}^{[t]} = \beta_1 v_{dW}^{[t-1]} + (1 - \beta_1) dW^{[t]}$$
$$v_{db}^{[t]} = \beta_1 v_{db}^{[t-1]} + (1 - \beta_1) db^{[t]}$$

### Step 2: Second Moment Estimation (Uncentered Gradient Variance)
The algorithm computes the EWA of the squared gradients (element-wise). This serves as a continuous measure of the gradient's amplitude and volatility:
$$s_{dW}^{[t]} = \beta_2 s_{dW}^{[t-1]} + (1 - \beta_2) (dW^{[t]})^2$$
$$s_{db}^{[t]} = \beta_2 s_{db}^{[t-1]} + (1 - \beta_2) (db^{[t]})^2$$

### Step 3: Bias Correction (The Critical Stabilizer)
Because the state matrices/vectors ($v, s$) are initialized to $0$, they are severely biased toward zero during the initial phase of training. This is particularly dangerous for the second moment when $\beta_2 \approx 1$. Without correction, dividing by a near-zero $s$ would cause gradient explosion. 
Bias correction is mathematically mandatory to recover the true estimates:
$$\hat{v}_{dW}^{[t]} = \frac{v_{dW}^{[t]}}{1 - \beta_1^t} \quad ; \quad \hat{v}_{db}^{[t]} = \frac{v_{db}^{[t]}}{1 - \beta_1^t}$$
$$\hat{s}_{dW}^{[t]} = \frac{s_{dW}^{[t]}}{1 - \beta_2^t} \quad ; \quad \hat{s}_{db}^{[t]} = \frac{s_{db}^{[t]}}{1 - \beta_2^t}$$

### Step 4: Parameter Update
The final step synthesizes the corrected components. The base learning rate $\alpha$ is scaled inversely by the square root of the corrected second moment, and directed by the corrected first moment:
$$W^{[t]} = W^{[t-1]} - \alpha \frac{\hat{v}_{dW}^{[t]}}{\sqrt{\hat{s}_{dW}^{[t]}} + \epsilon}$$
$$b^{[t]} = b^{[t-1]} - \alpha \frac{\hat{v}_{db}^{[t]}}{\sqrt{\hat{s}_{db}^{[t]}} + \epsilon}$$

*(Note: The division and the addition of $\epsilon$ are performed strictly element-wise).*

## 3. Statistical Interpretation of the Nomenclature

The term **"Adaptive Moment Estimation"** directly reflects the algorithm's statistical architecture:
* **First Moment:** The expected value (Mean) of a random variable. The variable $v$ estimates the mean of the gradient distribution.
* **Second Moment:** Inherently tied to the uncentered Variance. The variable $s$ estimates the variance of the gradient distribution.

By continuously "Estimating" these "Moments" to dynamically "Adapt" the optimization sequence, the algorithm achieves convergence properties far superior to standard gradient descent methods.

## 4. Hyperparameter Topography

Adam's dominance in Deep Learning research is largely attributed to the extraordinary robustness of its default hyperparameters. The following values, recommended by the original authors, perform optimally across a vast majority of architectures:

1. **$\alpha$ (Learning Rate):** The primary hyperparameter requiring empirical tuning. A standard starting point is **$0.001$**.
2. **$\beta_1$ (Momentum Decay):** The exponential decay rate for the first moment estimates. Recommended: **$0.9$** (equivalent to retaining the inertia of the last $\sim 10$ steps).
3. **$\beta_2$ (RMSprop Decay):** The exponential decay rate for the second-moment estimates. Recommended: **$0.999$** (equivalent to retaining the volatility of the last $\sim 1000$ steps). A value exceptionally close to $1$ ensures denominator stability.
4. **$\epsilon$ (Epsilon):** A microscopic scalar added for numerical stability to prevent division by zero. Recommended: **$10^{-8}$**. 

## 5. Practical Implementations

In modern computational environments, Adam is heavily abstracted, autonomously handling the initialization, accumulation, and bias correction of both moments.

### 5.1. PyTorch Implementation
~~~python
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Computational Graph Definition
model = nn.Linear(in_features=20, out_features=1)
criterion = nn.BCEWithLogitsLoss()

# 2. Adam Optimizer Initialization
# Note: PyTorch consolidates beta parameters into a 'betas' tuple
optimizer = optim.Adam(
    model.parameters(), 
    lr=0.001, 
    betas=(0.9, 0.999), 
    eps=1e-08,
    weight_decay=0 # Optional L2 Regularization parameter
)
~~~

### 5.2. TensorFlow / Keras Implementation
~~~python
import tensorflow as tf

# 1. Graph Architecture Construction
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=(20,))
])

# 2. Adam Optimizer Configuration (Explicit Parametrization)
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-07
)

model.compile(optimizer=optimizer, loss='binary_crossentropy')
~~~

# LEARNING RATE DECAY

## 1. The Optimization Dilemma and Core Motivation

In the training process of Deep Neural Networks, maintaining a constant global learning rate ($\alpha$) introduces a convergence dilemma. 

Although a large $\alpha$ allows the model to take extensive steps to rapidly traverse plateaus during the initial stages, it becomes a source of instability as the parameters approach the minimum. Due to the stochastic noise inherent in Mini-batch Gradient Descent, excessively large steps cause the parameters to overshoot or perpetually oscillate around the optimal basin without ever truly converging to the absolute minimum.

**Learning Rate Decay (or Scheduling)** resolves this issue by systematically reducing the step size over time. This mechanism enables the algorithm to progress rapidly during the early phases and execute highly precise micro-adjustments in the final epochs, strictly forcing the model to converge at the minimum.

## 2. Standard Mathematical Formulations (Decay Schedules)

In practice, various mathematical functions are utilized to decay the learning rate. Let $\alpha_0$ denote the initial learning rate, $t$ the current epoch (or iteration), and $k$ or $\gamma$ the decay hyperparameters:

### 2.1. Time-Based / Inverse Decay
This method decays the learning rate inversely proportional to the number of iterations, generating a smooth and steady asymptotic decay curve.
$$\alpha = \frac{\alpha_0}{1 + k \cdot t}$$
*(Where $k$ is the decay rate).*

### 2.2. Exponential Decay
This method aggressively compresses the learning rate exponentially. It is particularly effective for models that require rapid initial convergence followed by an immediate transition into a fine-tuning phase.
$$\alpha = \alpha_0 \cdot \gamma^t$$
*(Where $\gamma$ is a constant strictly less than $1$. For instance, $\gamma = 0.95$ implies the learning rate is reduced by 5% at each step).*

### 2.3. Step Decay / Piecewise Constant
Instead of a continuous reduction, the algorithm maintains a constant learning rate for a predefined interval and then drops it abruptly (typically dividing by 2 or 10). In industrial applications, engineers often monitor the loss function; when the loss reaches a plateau, this reduction step is triggered.
$$\alpha = \alpha_0 \cdot \gamma^{\lfloor \frac{t}{s} \rfloor}$$
*(Where $s$ is the step interval, e.g., $s=10$ means the rate decays once every 10 epochs).*

## 3. Theoretical Synergy: Why Adam Still Requires LR Decay

A critical theoretical question frequently arises: *If adaptive algorithms like RMSprop and Adam already automatically adjust the local learning rate for each parameter based on the gradient variance, is a global learning rate decay still necessary?*

The academic consensus is: **Absolutely necessary.**

Although adaptive optimizers excel at modulating the *relative scale* among parameters (effectively handling sparse data and pathological curvature), the global learning rate ($\alpha$) continues to act as the **"Absolute Upper Bound"** for every update step.

As the neural network approaches the end of its training lifecycle, the gradients derived from mini-batches exhibit high stochastic noise. If the global ceiling ($\alpha$) is not proactively lowered, Adam's denominator ($\sqrt{s_{dW}}$) will continuously amplify these noisy gradients, causing the model to thrash and lose precision at the micro-level.

**Conclusion:** Integrating an adaptive optimizer (such as Adam, to handle multidimensional navigation) with a learning rate decay schedule (to enforce asymptotic convergence in the final stages) remains the Gold Standard paradigm for training state-of-the-art Deep Learning models.